human lung cancer

导入相关包

In [29]:
import scanpy as sc
import pandas as pd
import numpy as np
from sklearn.metrics import (homogeneity_score,v_measure_score,adjusted_mutual_info_score,normalized_mutual_info_score,adjusted_rand_score,fowlkes_mallows_score)


In [30]:
repeat_times = 2
simple_path = '/home/cavin/jt/benchmark/data/hbc/s1_filtered_cells.h5ad'
celltype_col = 'cell_type'
cell_emb_col = 'X_scGPT'
save_path = f"/home/cavin/jt/benchmark/experiments/embedings/batch_integrate/scgpt_human_breast_cancer_emb.parquet"


读取嵌入

In [31]:
loaded_emb_df = pd.read_parquet(save_path)
adata = sc.read_h5ad(simple_path)
aligned_emb_df = loaded_emb_df.reindex(adata.obs_names)
adata

AnnData object with n_obs × n_vars = 281780 × 313
    obs: 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'batch', 'cell_type', 'n_genes'
    var: 'ensemble_id', 'type'
    obsm: 'spatial'

In [32]:
adata.obsm[cell_emb_col] = aligned_emb_df.to_numpy(dtype=np.float32)
adata

AnnData object with n_obs × n_vars = 281780 × 313
    obs: 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'batch', 'cell_type', 'n_genes'
    var: 'ensemble_id', 'type'
    obsm: 'spatial', 'X_scGPT'

In [33]:
res_list = [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1.0]
random_seed=2026 + repeat_times * 200
key_added='leiden'
sc.pp.neighbors(adata, use_rep=cell_emb_col,random_state=random_seed)
adata


AnnData object with n_obs × n_vars = 281780 × 313
    obs: 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'batch', 'cell_type', 'n_genes'
    var: 'ensemble_id', 'type'
    uns: 'neighbors'
    obsm: 'spatial', 'X_scGPT'
    obsp: 'distances', 'connectivities'

In [34]:
def main(adata,random_seed,res_list,key_added,celltype_col):
    max_value = 0
    metrics = {"method": "scGPT", "ARI": 0, "NMI": 0, "AMI": 0, "HS": 0, "VM": 0, "FMI": 0, "mean value": 0}
    save_label_df = None
    for used_res in res_list:
        sc.tl.leiden(adata, random_state=random_seed, resolution=used_res,key_added=key_added)
        true_labels = np.array(adata.obs[celltype_col])
        cluster_labels = np.array(adata.obs[key_added])
        FMI = fowlkes_mallows_score(true_labels, cluster_labels)
        homogeneity = homogeneity_score(true_labels, cluster_labels)
        v_measure = v_measure_score(true_labels, cluster_labels)
        ami = adjusted_mutual_info_score(true_labels, cluster_labels)
        nmi = normalized_mutual_info_score(true_labels, cluster_labels)
        ari = adjusted_rand_score(true_labels, cluster_labels)
        mean_value = np.mean([FMI, homogeneity, v_measure, ami, nmi, ari])

        n_cluster = len(adata.obs[key_added].unique())
        n_celltype = len(adata.obs[celltype_col].unique())
        if mean_value > max_value:
            save_label_df = adata.obs[key_added]
            metrics["ARI"] = ari
            metrics["NMI"] = nmi
            metrics["AMI"] = ami
            metrics["HS"] = homogeneity
            metrics["VM"] = v_measure
            metrics["FMI"] = FMI
            metrics["mean value"] = mean_value
            max_value = mean_value

        print(f'resolution = {used_res} | n_celltype = {n_celltype} | n_cluster = {n_cluster} | mean_value = {round(mean_value,3)} | ARI = {round(ari,3)} |  NMI = {round(nmi,3)} | AMI = {round(ami,3)} | HS = {round(homogeneity,3)} | VM = {round(v_measure,3)} | FMI = {round(FMI,3)} ')

    return metrics, save_label_df

In [35]:
metrics, save_label_df = main(adata,random_seed,res_list,key_added,celltype_col)

resolution = 0.1 | n_celltype = 20 | n_cluster = 3 | mean_value = 0.365 | ARI = 0.249 |  NMI = 0.394 | AMI = 0.394 | HS = 0.265 | VM = 0.394 | FMI = 0.492 
resolution = 0.2 | n_celltype = 20 | n_cluster = 7 | mean_value = 0.491 | ARI = 0.373 |  NMI = 0.538 | AMI = 0.538 | HS = 0.452 | VM = 0.538 | FMI = 0.504 
resolution = 0.3 | n_celltype = 20 | n_cluster = 7 | mean_value = 0.467 | ARI = 0.357 |  NMI = 0.51 | AMI = 0.51 | HS = 0.439 | VM = 0.51 | FMI = 0.476 
resolution = 0.4 | n_celltype = 20 | n_cluster = 11 | mean_value = 0.461 | ARI = 0.302 |  NMI = 0.522 | AMI = 0.522 | HS = 0.499 | VM = 0.522 | FMI = 0.399 
resolution = 0.5 | n_celltype = 20 | n_cluster = 11 | mean_value = 0.456 | ARI = 0.298 |  NMI = 0.516 | AMI = 0.516 | HS = 0.493 | VM = 0.516 | FMI = 0.396 
resolution = 0.6 | n_celltype = 20 | n_cluster = 12 | mean_value = 0.456 | ARI = 0.297 |  NMI = 0.517 | AMI = 0.517 | HS = 0.496 | VM = 0.517 | FMI = 0.395 
resolution = 0.7 | n_celltype = 20 | n_cluster = 13 | mean_value

In [36]:
metrics

{'method': 'scGPT',
 'ARI': 0.3729860773290563,
 'NMI': 0.5381298215620914,
 'AMI': 0.5380821638174091,
 'HS': 0.45245165005218524,
 'VM': 0.5381298215620914,
 'FMI': 0.50388527239771,
 'mean value': 0.49061080112009064}

In [37]:
save_label_df

s1r1_1         0
s1r1_2         0
s1r1_5         1
s1r1_8         0
s1r1_9         1
              ..
s1r2_118748    3
s1r2_118749    3
s1r2_118750    3
s1r2_118751    3
s1r2_118752    3
Name: leiden, Length: 281780, dtype: category
Categories (7, object): ['0', '1', '2', '3', '4', '5', '6']

In [38]:
save_label_df_path = f"/home/cavin/jt/benchmark/experiments/results/labels_df/breast_cancer/scgpt_human_breast_cancer_labels_repeat_{repeat_times}.csv"

In [39]:
save_label_df.to_csv(save_label_df_path)

In [40]:
metrics_df = pd.DataFrame.from_dict(metrics, orient='index').T

In [41]:
metrics_df_save_path = f"/home/cavin/jt/benchmark/experiments/results/cluster_metrics/breast_cancer/human_breast_cancer_metrics_repeat_{repeat_times}.csv"

In [42]:
metrics_df.to_csv(metrics_df_save_path, index=False,mode="a", header=not pd.io.common.file_exists(metrics_df_save_path))

In [1]:
import pandas as pd
df0 = pd.read_csv("/home/cavin/jt/benchmark/experiments/results/cluster_metrics/breast_cancer/human_breast_cancer_metrics_repeat_0.csv",index_col=0,header=0)
df1 = pd.read_csv("/home/cavin/jt/benchmark/experiments/results/cluster_metrics/breast_cancer/human_breast_cancer_metrics_repeat_1.csv",index_col=0,header=0)
df2 = pd.read_csv("/home/cavin/jt/benchmark/experiments/results/cluster_metrics/breast_cancer/human_breast_cancer_metrics_repeat_2.csv",index_col=0,header=0)

In [2]:
df0

,ARI,NMI,AMI,HS,VM,FMI,mean value
method,,,,,,,
OmniCell,0.443169,0.613644,0.613583,0.565590,0.613644,0.526079,0.562618
Nicheformer,0.243177,0.432661,0.432547,0.438747,0.432661,0.338345,0.386356
Harmony,0.560955,0.625696,0.625654,0.561232,0.625696,0.635637,0.605812
Geneformer,0.000139,0.000386,0.000364,0.000249,0.000386,0.268128,0.044942
scGPT,0.349643,0.529875,0.529804,0.509379,0.529875,0.437418,0.480999
scFoundation,0.353422,0.484967,0.484915,0.417971,0.484967,0.475102,0.450224
PCA,0.569696,0.618658,0.618616,0.564688,0.618658,0.635448,0.604294


In [3]:
df1

,ARI,NMI,AMI,HS,VM,FMI,mean value
method,,,,,,,
OmniCell,0.484607,0.621577,0.621517,0.573388,0.621577,0.561108,0.580629
Geneformer,0.000136,0.000704,0.000659,0.000457,0.000704,0.267863,0.045087
Nicheformer,0.283833,0.422981,0.422883,0.398981,0.422981,0.391188,0.390474
Harmony,0.565861,0.627926,0.627863,0.595210,0.627926,0.626978,0.611961
scFoundation,0.323548,0.462105,0.462050,0.392833,0.462105,0.455318,0.426326
scGPT,0.358523,0.552774,0.552715,0.489839,0.552774,0.473341,0.496661
PCA,0.539637,0.605864,0.605820,0.554690,0.605864,0.608594,0.586745


In [4]:
df2

,ARI,NMI,AMI,HS,VM,FMI,mean value
method,,,,,,,
OmniCell,0.456335,0.610920,0.610860,0.573162,0.610920,0.533414,0.565935
Harmony,0.488183,0.623701,0.623633,0.611065,0.623701,0.555711,0.587666
Geneformer,0.000118,0.000409,0.000364,0.000266,0.000409,0.267124,0.044782
Nicheformer,0.239164,0.422738,0.422623,0.432482,0.422738,0.333646,0.378899
PCA,0.535695,0.618644,0.618597,0.576719,0.618644,0.603425,0.595287
scGPT,0.372986,0.538130,0.538082,0.452452,0.538130,0.503885,0.490611
scFoundation,0.314858,0.465291,0.465231,0.420553,0.465291,0.423827,0.425842


In [5]:
df = (df0+df1+df2)
df

,ARI,NMI,AMI,HS,VM,FMI,mean value
method,,,,,,,
Geneformer,0.000393,0.001499,0.001386,0.000972,0.001499,0.803115,0.134811
Harmony,1.614999,1.877323,1.877151,1.767507,1.877323,1.818326,1.805438
Nicheformer,0.766174,1.278379,1.278053,1.270210,1.278379,1.063179,1.155729
OmniCell,1.384112,1.846141,1.845959,1.712140,1.846141,1.620601,1.709182
PCA,1.645027,1.843166,1.843033,1.696097,1.843166,1.847467,1.786326
scFoundation,0.991827,1.412363,1.412196,1.231357,1.412363,1.354247,1.302392
scGPT,1.081152,1.620779,1.620602,1.451670,1.620779,1.414644,1.468271


In [8]:
df = df / 3
df

,ARI,NMI,AMI,HS,VM,FMI,mean value
method,,,,,,,
Geneformer,0.000131,0.000500,0.000462,0.000324,0.000500,0.267705,0.044937
Harmony,0.538333,0.625774,0.625717,0.589169,0.625774,0.606109,0.601813
Nicheformer,0.255391,0.426126,0.426018,0.423403,0.426126,0.354393,0.385243
OmniCell,0.461371,0.615380,0.615320,0.570713,0.615380,0.540200,0.569727
PCA,0.548342,0.614389,0.614344,0.565366,0.614389,0.615822,0.595442
scFoundation,0.330609,0.470788,0.470732,0.410452,0.470788,0.451416,0.434131
scGPT,0.360384,0.540260,0.540201,0.483890,0.540260,0.471548,0.489424


In [9]:
df.to_csv("/home/cavin/jt/benchmark/experiments/results/cluster_metrics/breast_cancer/human_breast_cancer_metrics_mean.csv")